# Step 3 — Final Training


In [ ]:
import os
import time
import torch
import numpy as np
import pandas as pd
import random
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data 
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import sys


In [ ]:
# --- Paths and Configuration ---
BASE_OUTPUT_DIR = './output' 
DATA_FILENAME = 'processed_graph_data.pt'
METADATA_FILENAME = 'metadata.pt'
RESULTS_FILENAME = 'optuna_study_results.csv' 

data_path = os.path.join(BASE_OUTPUT_DIR, DATA_FILENAME)
metadata_path = os.path.join(BASE_OUTPUT_DIR, METADATA_FILENAME)
results_path = os.path.join(BASE_OUTPUT_DIR, RESULTS_FILENAME)


# --- Seed Function ---
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(42)

# --- Load Data and Hyperparameters ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
map_loc = 'cpu' if device.type == 'cpu' else None

print(f"Dispositivo de Cómputo: {device}")

data = None
metadata = None
IN_CHANNELS = None
# Initialize variables for use after try blocks
protein_to_idx = None 
idx_to_protein = None 

# First Try: Load Data
try:
    data = torch.load(data_path, map_location=map_loc, weights_only=False) 
    metadata = torch.load(metadata_path, weights_only=False)
    IN_CHANNELS = data.x.shape[1]
    
    # === Extract KEY Mappings ===
    if 'protein_to_idx' in metadata and 'idx_to_protein' in metadata:
        protein_to_idx = metadata['protein_to_idx']
        idx_to_protein = metadata['idx_to_protein'] # El diccionario inverso podría ser útil
        print(f"  Protein mappings (protein_to_idx) successfully extracted.")
    else:
        print("🚨 Warning: Protein mappings not found in metadata.")

    print(f"✔️ Graph and metadata successfully loaded. IN_CHANNELS: {IN_CHANNELS}")
except Exception as e:
    print(f"🚨 Error loading data: {e}")

if data is not None:
    # Second Try: Load Optimal Hyperparameters (CRITICAL)
    try:
        df_results = pd.read_csv(results_path)
        best_trial_row = df_results.loc[df_results['value'].idxmax()]
        
        # Extraer HPs 
        HIDDEN_CHANNELS = int(best_trial_row['params_hidden_channels'])
        OUT_CHANNELS = int(best_trial_row['params_out_channels'])
        NUM_HEADS = int(best_trial_row['params_num_heads'])
        PREDICTOR_HIDDEN_CHANNELS = int(best_trial_row['params_predictor_hidden_channels'])
        LEARNING_RATE = float(best_trial_row['params_learning_rate'])
        DROPOUT_RATE = float(best_trial_row['params_dropout_rate'])
        ACTIVATION_FN_NAME = best_trial_row['params_activation_function']
        FINAL_EPOCHS = int(best_trial_row['params_epochs']) 

        print("\n✅ Optimal Hyperparameters Loaded:")
        print(f"  Hidden Channels: {HIDDEN_CHANNELS}, Output Channels: {OUT_CHANNELS}, Heads: {NUM_HEADS}")
        print(f"  LR: {LEARNING_RATE}, Dropout: {DROPOUT_RATE}, Epochs: {FINAL_EPOCHS}")
        
    except Exception as e:
        # If loading HPs fails, print the error and stop execution.
        # Since HPs and 'data' are not reassigned, following cells will fail (NameError/AttributeError).
        print(f"\n❌ CRITICAL ERROR while loading optimal HPs: {e}")
        print("   >>> The program will stop here. Please check 'output/optuna_study_results.csv'.")
        raise SystemExit(1)  # Abort execution of the cell if necessary

## model architecture

In [ ]:
class GNNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_heads=1, add_self_loops=True, dropout_rate=0.0, activation_fn_name="relu"):
        super(GNNEncoder, self).__init__()
        
        self.conv1 = GATConv(in_channels, hidden_channels, heads=num_heads, 
                              add_self_loops=add_self_loops, edge_dim=1, dropout=dropout_rate)
        
        self.conv2 = GATConv(hidden_channels * num_heads, out_channels, heads=1, 
                              add_self_loops=add_self_loops, edge_dim=1, concat=False, dropout=dropout_rate)
        
        self.dropout_layer = nn.Dropout(dropout_rate) 

        if activation_fn_name == "relu":
            self.activation_fn = F.relu
        elif activation_fn_name == "tanh":
            self.activation_fn = F.tanh
        else:
            raise ValueError(f"Unsupported activation function '{activation_fn_name}'.")

    def forward(self, x, edge_index, edge_attr):
        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = self.activation_fn(x)  
        x = self.dropout_layer(x)  
        
        x = self.conv2(x, edge_index, edge_attr=edge_attr)
        return x

class LinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(LinkPredictor, self).__init__()
        self.lin1 = nn.Linear(in_channels * 2, hidden_channels) 
        self.lin2 = nn.Linear(hidden_channels, out_channels) 

    def forward(self, x_i, x_j):
        x = torch.cat([x_i, x_j], dim=-1) 
        x = self.lin1(x)
        x = F.relu(x) 
        x = self.lin2(x)
        return x

print("✔️ Classes GNNEncoder and LinkPredictor defined.")

## training evaluation

In [ ]:
def train(model, predictor, data, optimizer, criterion):
    model.train()
    predictor.train()
    optimizer.zero_grad()

    z = model(data.x, data.edge_index, data.edge_attr)

    # Positive edge predictions (train)
    pos_edge_index = data.train_pos_edge_index
    pos_pred = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])

    # Negative edge predictions (train)
    neg_edge_index = data.train_neg_edge_index
    neg_pred = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])

    # Concatenate logits and build targets
    pred = torch.cat([pos_pred, neg_pred], dim=0)
    target = torch.cat([torch.ones(pos_pred.size(0), device=pred.device), torch.zeros(neg_pred.size(0), device=pred.device)], dim=0)

    # Loss + backward
    train_loss = criterion(pred.squeeze(), target)
    train_loss.backward()
    optimizer.step()

    # Metrics on CPU
    pred_cpu = pred.detach().cpu().numpy().squeeze()
    target_cpu = target.cpu().numpy()
    preds_bin = (pred_cpu >= 0.0).astype(int) 

    train_auc = roc_auc_score(target_cpu, pred_cpu)
    train_acc = accuracy_score(target_cpu, preds_bin)
    train_precision = precision_score(target_cpu, preds_bin, zero_division=0)
    train_recall = recall_score(target_cpu, preds_bin, zero_division=0)
    train_f1 = f1_score(target_cpu, preds_bin, zero_division=0)

    return train_loss.item(), train_auc, train_acc, train_precision, train_recall, train_f1

@torch.no_grad() 
def test(model, predictor, data):
    model.eval()
    predictor.eval()

    z = model(data.x, data.edge_index, data.edge_attr)

    # Helper to get logits and targets for a split
    def get_preds_targets(pos_edge_index, neg_edge_index):
        pos_pred = predictor(z[pos_edge_index[0]], z[pos_edge_index[1]])
        neg_pred = predictor(z[neg_edge_index[0]], z[neg_edge_index[1]])
        
        preds_tensor = torch.cat([pos_pred, neg_pred], dim=0).squeeze()
        targets_tensor = torch.cat([torch.ones(pos_pred.size(0), device=preds_tensor.device), torch.zeros(neg_pred.size(0), device=preds_tensor.device)], dim=0)
        
        return preds_tensor.cpu().numpy(), targets_tensor.cpu().numpy()

    # Validation
    val_preds, val_targets = get_preds_targets(data.val_pos_edge_index, data.val_neg_edge_index)

    # Test
    test_preds, test_targets = get_preds_targets(data.test_pos_edge_index, data.test_neg_edge_index)

    # Loss on CPU with logits (BCEWithLogits expects raw scores)
    val_loss = F.binary_cross_entropy_with_logits(torch.tensor(val_preds), torch.tensor(val_targets))
    test_loss = F.binary_cross_entropy_with_logits(torch.tensor(test_preds), torch.tensor(test_targets))

    # Calculate Metrics
    val_auc = roc_auc_score(val_targets, val_preds)
    test_auc = roc_auc_score(test_targets, test_preds) 

    val_preds_bin = (val_preds >= 0.0).astype(int) 
    test_preds_bin = (test_preds >= 0.0).astype(int)

    val_acc = accuracy_score(val_targets, val_preds_bin)
    test_acc = accuracy_score(test_targets, test_preds_bin)
    val_precision = precision_score(val_targets, val_preds_bin, zero_division=0)
    test_precision = precision_score(test_targets, test_preds_bin, zero_division=0)
    val_recall = recall_score(val_targets, val_preds_bin, zero_division=0)
    test_recall = recall_score(test_targets, test_preds_bin, zero_division=0)
    val_f1 = f1_score(val_targets, val_preds_bin, zero_division=0)
    test_f1 = f1_score(test_targets, test_preds_bin, zero_division=0)

    return val_loss, test_loss, val_auc, test_auc, val_acc, test_acc, val_precision, test_precision, val_recall, test_recall, val_f1, test_f1

print("✔️ Training and evaluation functions defined.")

## División de enlaces

In [ ]:
if data is not None:
    set_seed(42)  # Using the same seed is CRUCIAL to replicate the split

    print("\nSplitting final edges for training/validation/testing...")

    # RandomLinkSplit must be identical to the one used in 02_tuning
    transform = RandomLinkSplit(
        num_val=0.1,
        num_test=0.1,
        is_undirected=True,
        add_negative_train_samples=True,
        split_labels=True
    )
    
    train_data, val_data, test_data = transform(data) 
    # Create the final data object that the GNN will use for propagation (train edges only)
    data_final = Data(x=train_data.x, edge_index=train_data.edge_index, edge_attr=train_data.edge_attr)
    
    # Assign edge indices for the prediction function
    data_final.train_pos_edge_index = train_data.pos_edge_label_index
    data_final.train_neg_edge_index = train_data.neg_edge_label_index
    data_final.val_pos_edge_index   = val_data.pos_edge_label_index
    data_final.val_neg_edge_index   = val_data.neg_edge_label_index
    data_final.test_pos_edge_index  = test_data.pos_edge_label_index
    data_final.test_neg_edge_index  = test_data.neg_edge_label_index

    # Move everything to the selected compute device
    data_final = data_final.to(device)

    print(f"  Total Nodes: {data_final.x.shape[0]}")
    print(f"  Total Test Positive Edges: {data_final.test_pos_edge_index.shape[1]}")
    print("✔️ Final data split completed.")

In [ ]:
if 'data_final' in locals() and data_final is not None:
    set_seed(42)
    start_time = time.time()
    
    data_final = data_final.to(device)
    
    # 1. Inicialización del Modelo con HPs cargados
    print(f"Initializing GNNEncoder (in_channels={IN_CHANNELS}, hidden_channels={HIDDEN_CHANNELS}, out_channels={OUT_CHANNELS}, num_heads={NUM_HEADS})...")
    
   # Use HIDDEN_CHANNELS, OUT_CHANNELS, NUM_HEADS, DROPOUT_RATE, ACTIVATION_FN_NAME
    model = GNNEncoder(IN_CHANNELS, HIDDEN_CHANNELS, OUT_CHANNELS, 
                       num_heads=NUM_HEADS, dropout_rate=DROPOUT_RATE, 
                       activation_fn_name=ACTIVATION_FN_NAME).to(device)
    
    # Use PREDICTOR_HIDDEN_CHANNELS
    predictor = LinkPredictor(OUT_CHANNELS, PREDICTOR_HIDDEN_CHANNELS, 1).to(device) 

    optimizer = torch.optim.Adam(list(model.parameters()) + list(predictor.parameters()), lr=LEARNING_RATE)
    criterion = torch.nn.BCEWithLogitsLoss() 

    results = []
    best_val_auc = 0.0 
    best_epoch = 0
    best_model_state = None 
    best_predictor_state = None 
    
    # Use FINAL_EPOCHS loaded from the CSV
    EPOCHS = FINAL_EPOCHS
    
    print(f"Starting training for {EPOCHS} epochs...")
    
    for epoch in range(1, EPOCHS + 1):
        # NOTE: We use 'data_final' instead of 'data' for training,
        # because 'data_final' contains the split indices.
        train_loss, train_auc, train_acc, train_precision, train_recall, train_f1 = train(model, predictor, data_final, optimizer, criterion)
        val_loss, test_loss, val_auc, test_auc, val_acc, test_acc, val_precision, test_precision, val_recall, test_recall, val_f1, test_f1 = test(model, predictor, data_final)

        # Append metrics (keeps your original schema)
        results.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,            # already .item() inside test()
            'test_loss': test_loss,          # already .item() inside test()
            'train_auc': train_auc,
            'val_auc': val_auc,
            'test_auc': test_auc,
            'train_acc': train_acc,
            'val_acc': val_acc,
            'test_acc': test_acc,
            'train_precision': train_precision,
            'val_precision': val_precision,
            'test_precision': test_precision,
            'train_recall': train_recall,
            'val_recall': val_recall,
            'test_recall': test_recall,
            'train_f1': train_f1,
            'val_f1': val_f1,
            'test_f1': test_f1
        })
        
        # Track best validation AUC
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            # Keep a copy of the best model/predictor states
            best_model_state = model.state_dict()
            best_predictor_state = predictor.state_dict()
            print(f"    ⭐ New best validation AUC at epoch {epoch}: {best_val_auc:.4f}")

        
        #  Print metrics every 10 epochs (and at start/end)
        if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
            print(f'  Epoch: {epoch:03d} | '
                f'Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Test Loss: {test_loss:.4f} | '
                f'Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f} | Test AUC: {test_auc:.4f}')

    # Save metrics to CSV (FULL training report)
    output_path = os.path.join(BASE_OUTPUT_DIR, "training_metrics_results.csv")
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_path, index=False)
    print(f"\nAll-epoch metrics saved to: {output_path}")

    print("\nModel training completed.")
    print(f"Best validation epoch: {best_epoch} with AUC = {best_val_auc:.4f}")

    if best_model_state:
        # Load best model weights
        model.load_state_dict(best_model_state)
        predictor.load_state_dict(best_predictor_state)
        print(f"Loading best epoch ({best_epoch}) weights to generate final embeddings.")


        # Generate final embeddings
        print("Generating final embeddings with the best model...")
        model.eval()

        with torch.no_grad():
            # Usamos data_final ya que tiene las features (data.x) y la estructura de entrenamiento
            final_embeddings = model(data_final.x, data_final.edge_index, data_final.edge_attr).cpu().numpy()
        print(f"Final embeddings generated. Shape: {final_embeddings.shape}")

        # Save final embeddings to CSV
        # NOTE: We need node ID mapping. If 'metadata' contains it or the order of
        # data_final.x matches the original mapping, we can use protein_to_idx keys.

        # Generate the list of protein IDs in the correct order
        if protein_to_idx is not None:
            # list(protein_to_idx.keys()) preserves node order
            proteins_ids = list(protein_to_idx.keys()) 
            emb_df = pd.DataFrame(final_embeddings, index=proteins_ids)

            emb_output_path = os.path.join(BASE_OUTPUT_DIR, "embeddings.csv")
            emb_df.to_csv(emb_output_path, index=True)  # index=True keeps protein IDs as first column
            print(f"✅ Embeddings saved with protein names as the first column at: {emb_output_path}")
        else:
            print("🚨 ERROR: 'protein_to_idx' is not available. Embeddings CSV was not saved.")

        # Save to .csv file
        emb_output_path = os.path.join(BASE_OUTPUT_DIR, "embeddings.csv")
        emb_df.to_csv(emb_output_path)  # Save without pandas index
        print(f"[INFO] Embeddings saved at: {emb_output_path}")

    # --- Execution Time ---
    end_time = time.time()
    execution_time = end_time - start_time
    print("\n--- Pipeline Completed ---")
    print(f"Total execution time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)")
    print("Analysis finished successfully!")
    
else:
    print("🚨 ERROR: 'data_final' is not available.")